# Chassis Impact Detector -- Stage 3 Training (Kaggle)

Trains the lightweight CNN on the log-mel spectrogram windows produced by Stage 2
(`ai_audio/preprocessing/make_features.py`, run on the Pi), then exports the result
to ONNX for Stage 4 (NCNN inference back on the Pi).

**Before running:**
1. On the Pi: `cd ai_audio/dataset && zip -r processed_dataset.zip processed`
2. Upload `processed_dataset.zip` as a new Kaggle Dataset (kaggle.com -> Datasets -> New Dataset).
3. Attach that dataset to this notebook (right sidebar -> Add Input).
4. Turn on a GPU accelerator: Settings -> Accelerator -> GPU T4 x2 (or P100).
5. Set `INPUT_DIR` in the next config cell to match your attached dataset's mount path.


In [ ]:
import csv, json, os, random
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())


In [ ]:
# Kaggle mounts uploaded datasets under /kaggle/input/<dataset-slug>/...
# Adjust this to wherever your uploaded processed_dataset.zip landed.
INPUT_DIR = "/kaggle/input/chassis-impact-processed/processed"
OUTPUT_DIR = "/kaggle/working"

# Raw labels recorded on the Pi that should collapse into a single training
# class. Add an entry here whenever you record a new hard-negative session
# under its own label (e.g. driving + self-fire noise) instead of directly
# under "background" -- this keeps the raw data traceable (you can still
# measure false-positive rate per source) while training stays a plain
# binary hit/background classifier.
LABEL_MAP = {
    "background_self_fire": "background",
}

VAL_FRAC = 0.2
EPOCHS = 40
BATCH_SIZE = 16
LR = 1e-3
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:
def load_manifest(processed_dir):
    with open(os.path.join(processed_dir, "manifest.csv")) as f:
        rows = list(csv.DictReader(f))
    for r in rows:
        r["label"] = LABEL_MAP.get(r["label"], r["label"])
    return rows

rows = load_manifest(INPUT_DIR)
labels = sorted(set(r["label"] for r in rows))
label_to_idx = {label: i for i, label in enumerate(labels)}
print("labels:", label_to_idx)
print("total windows:", len(rows))
for label in labels:
    print(" ", label, sum(1 for r in rows if r["label"] == label))


In [ ]:
def group_split(rows, val_frac, seed):
    """Hold out whole clips (source_wav) for validation, stratified by label,
    so windows sliced from the same clip never end up split across train/val."""
    by_label_clip = defaultdict(set)
    for r in rows:
        by_label_clip[r["label"]].add(r["source_wav"])

    rng = random.Random(seed)
    val_clips = set()
    for label, clips in by_label_clip.items():
        clips = sorted(clips)
        rng.shuffle(clips)
        n_val = max(1, round(len(clips) * val_frac))
        val_clips.update(clips[:n_val])

    train_rows = [r for r in rows if r["source_wav"] not in val_clips]
    val_rows = [r for r in rows if r["source_wav"] in val_clips]
    return train_rows, val_rows

train_rows, val_rows = group_split(rows, VAL_FRAC, SEED)
print(f"train windows: {len(train_rows)}  val windows: {len(val_rows)}")


In [ ]:
def compute_stats(rows, processed_dir):
    vals = [np.load(os.path.join(processed_dir, r["path"])) for r in rows]
    arr = np.stack(vals)
    return float(arr.mean()), float(arr.std())

mean, std = compute_stats(train_rows, INPUT_DIR)
print(f"train feature mean={mean:.3f} std={std:.3f}")

def spec_augment(feat, freq_mask=8, time_mask=12):
    feat = feat.copy()
    n_mels, n_frames = feat.shape
    f0 = random.randint(0, max(0, n_mels - freq_mask))
    feat[f0:f0 + freq_mask, :] = 0.0
    t0 = random.randint(0, max(0, n_frames - time_mask))
    feat[:, t0:t0 + time_mask] = 0.0
    return feat

class SpecDataset(Dataset):
    def __init__(self, rows, processed_dir, label_to_idx, mean, std, augment):
        self.rows = rows
        self.processed_dir = processed_dir
        self.label_to_idx = label_to_idx
        self.mean = mean
        self.std = std
        self.augment = augment

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        feat = np.load(os.path.join(self.processed_dir, row["path"]))
        feat = (feat - self.mean) / self.std
        if self.augment:
            feat = spec_augment(feat)
        x = torch.from_numpy(feat).float().unsqueeze(0)
        y = self.label_to_idx[row["label"]]
        return x, y

train_ds = SpecDataset(train_rows, INPUT_DIR, label_to_idx, mean, std, augment=True)
val_ds = SpecDataset(val_rows, INPUT_DIR, label_to_idx, mean, std, augment=False)

class_counts = defaultdict(int)
for r in train_rows:
    class_counts[r["label"]] += 1
sample_weights = [1.0 / class_counts[r["label"]] for r in train_rows]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_rows), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

sample_x, _ = train_ds[0]
print("model input shape:", tuple(sample_x.shape))


In [ ]:
class ImpactCNN(nn.Module):
    def __init__(self, n_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(64, n_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ImpactCNN(n_classes=len(labels)).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"device: {device}, params: {n_params:,}")


In [ ]:
def evaluate(model, loader, device):
    model.eval()
    tp = tn = fp = fn = 0
    hit_idx = label_to_idx.get("hit", 1)
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            for p, t in zip(pred.tolist(), y.tolist()):
                tp += p == hit_idx and t == hit_idx
                tn += p != hit_idx and t != hit_idx
                fp += p == hit_idx and t != hit_idx
                fn += p != hit_idx and t == hit_idx
    total = tp + tn + fp + fn
    accuracy = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1,
            "tp": tp, "tn": tn, "fp": fp, "fn": fn}


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

os.makedirs(OUTPUT_DIR, exist_ok=True)
best_f1 = -1.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    train_loss = total_loss / len(train_ds)

    metrics = evaluate(model, val_loader, device)
    print(
        f"epoch {epoch:3d}  train_loss={train_loss:.4f}  "
        f"val_acc={metrics['accuracy']:.3f}  val_f1={metrics['f1']:.3f}  "
        f"(tp={metrics['tp']} fp={metrics['fp']} fn={metrics['fn']} tn={metrics['tn']})"
    )

    if metrics["f1"] >= best_f1:
        best_f1 = metrics["f1"]
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
print(f"best val f1={best_f1:.3f}")


In [ ]:
model.eval().cpu()
sample_shape = tuple(sample_x.shape)  # (1, n_mels, n_frames)
dummy_input = torch.randn(1, *sample_shape)

onnx_path = os.path.join(OUTPUT_DIR, "impact_cnn.onnx")
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=["spectrogram"],
    output_names=["logits"],
    opset_version=12,
    dynamic_axes=None,  # fixed shape on purpose -- must match Stage 4's window length exactly
)
print("exported", onnx_path)

with open(os.path.join(OUTPUT_DIR, "labels.json"), "w") as f:
    json.dump(label_to_idx, f, indent=2)
with open(os.path.join(OUTPUT_DIR, "feature_stats.json"), "w") as f:
    json.dump({"mean": mean, "std": std, "input_shape": list(sample_shape)}, f, indent=2)

print("labels.json and feature_stats.json written to", OUTPUT_DIR)


In [ ]:
import onnxruntime as ort

sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
with torch.no_grad():
    torch_out = model(dummy_input).numpy()
onnx_out = sess.run(None, {"spectrogram": dummy_input.numpy()})[0]

max_diff = np.abs(torch_out - onnx_out).max()
print("max abs diff torch vs onnx:", max_diff)
assert max_diff < 1e-4, "ONNX export mismatch!"
print("ONNX export verified.")


## Done

Download these three files from the notebook's **Output** panel (`/kaggle/working`) and copy them
to `ai_audio/model/` on the Pi for Stage 4:

- `impact_cnn.onnx`
- `labels.json`
- `feature_stats.json`
